In [1]:
# ============================================================
# SIMPLE AI BITCOIN PAPER TRADING BOT
# ============================================================
#
# What this program does:
#
# 1. Gets Bitcoin price data
# 2. Calculates RSI and MACD
# 3. Uses those indicators to decide BUY / SELL / HOLD
# 4. Automatically executes simulated trades
# 5. Starts with $1,000 of fake money
# 6. Shows the portfolio and profit/loss after every check
#
# IMPORTANT:
# This is a PAPER TRADING bot.
# No real money is being traded.
#
# Install the libraries first:
#
# pip install yfinance pandas ta
#
# Then run:
#
# python bitcoin_bot.py
#
# ============================================================


# Import Python's built-in time module.
# We use this module later to make the bot wait between market checks.
import time

# Import the yfinance library and give it the shorter name "yf".
# yfinance allows us to download financial market data.
import yfinance as yf

# Import pandas and give it the shorter name "pd".
# pandas allows us to store and work with our market data in table form.
import pandas as pd

# Import RSIIndicator from the "ta" technical analysis library.
# This gives us a ready-made function for calculating RSI.
from ta.momentum import RSIIndicator

# Import MACD from the "ta" technical analysis library.
# This gives us a ready-made function for calculating MACD.
from ta.trend import MACD


# ============================================================
# SETTINGS
# ============================================================

# Starting paper trading money.
# This is fake money because this program is only simulating trades.
STARTING_MONEY = 1000.00

# Bitcoin ticker symbol.
# BTC-USD means Bitcoin priced in US dollars.
SYMBOL = "BTC-USD"

# How often the bot checks the market.
# 60 seconds = 1 minute.
# We can change this number to control how frequently the bot checks.
CHECK_EVERY_SECONDS = 60

# How much of our available cash we use when buying.
# 0.25 means 25%.
#
# For example:
# If we have $1,000:
# $1,000 x 0.25 = $250
#
# Therefore, the bot would use $250 for the simulated purchase.
TRADE_SIZE = 0.25


# ============================================================
# PAPER TRADING ACCOUNT
# ============================================================

# Create our current cash balance.
# We start with the amount stored in STARTING_MONEY.
cash = STARTING_MONEY

# Create our current Bitcoin balance.
# We start with zero Bitcoin because our paper account
# does not own any BTC when the program begins.
btc = 0.0

# Create an empty list that will store our trade history.
#
# Every time the bot buys or sells, we will add information
# about that trade to this list.
trade_history = []


# ============================================================
# GET BITCOIN DATA
# ============================================================

# Define a function called get_bitcoin_data().
#
# A function is a reusable block of code.
# Instead of writing the same market-data code repeatedly,
# we can simply call:
#
# get_bitcoin_data()
#
# whenever we need new Bitcoin data.
def get_bitcoin_data():

    # Download the most recent Bitcoin market data.

    # We use yfinance's download() function to retrieve the data.
    data = yf.download(

        # Tell yfinance which asset we want.
        # SYMBOL currently contains "BTC-USD".
        SYMBOL,

        # Download the most recent 5 days of data.
        period="5d",

        # Each candle represents 5 minutes.
        # Therefore, every row represents a 5-minute period.
        interval="5m",

        # Do not show yfinance's progress bar.
        # This keeps our terminal output cleaner.
        progress=False,

        # Do not automatically adjust the downloaded prices.
        auto_adjust=False
    )

    # Sometimes yfinance returns columns with multiple levels.
    # pandas calls this structure a MultiIndex.
    #
    # We want simpler column names such as:
    #
    # Open
    # High
    # Low
    # Close
    # Volume
    #
    # So we check whether the columns are a MultiIndex.
    if isinstance(data.columns, pd.MultiIndex):

        # If they are, keep only the first level of the column names.
        # This makes it easier for us to access:
        #
        # data["Close"]
        #
        # instead of dealing with multiple column levels.
        data.columns = data.columns.get_level_values(0)

    # Return the completed market-data table.
    #
    # Whatever code calls get_bitcoin_data()
    # will receive this DataFrame.
    return data


# ============================================================
# CALCULATE RSI AND MACD
# ============================================================

# Define a function called calculate_indicators().
#
# The function receives our market data as "data".
#
# It will calculate RSI and MACD and add them
# as new columns to the DataFrame.
def calculate_indicators(data):

    # --------------------------------------------------------
    # RSI
    # --------------------------------------------------------

    # RSI stands for Relative Strength Index.
    #
    # RSI is a technical indicator that generally ranges
    # from 0 to 100.
    #
    # A common interpretation is:
    #
    # Below 30 = potentially oversold
    # Above 70 = potentially overbought
    #
    # These levels are not guarantees that the price will
    # reverse. They are simply signals we are using
    # for this teaching example.

    # Create an RSIIndicator object.
    #
    # We give it the Bitcoin closing prices because RSI
    # is calculated using the closing-price series.
    rsi_indicator = RSIIndicator(

        # Tell RSIIndicator to use the "Close" column.
        close=data["Close"],

        # Use 14 periods for the RSI calculation.
        # Because our candles are 5 minutes,
        # this represents 14 five-minute candles.
        window=14
    )

    # Calculate the RSI values.
    #
    # We create a new column called "RSI"
    # and store the calculated RSI values there.
    data["RSI"] = rsi_indicator.rsi()


    # --------------------------------------------------------
    # MACD
    # --------------------------------------------------------

    # MACD stands for Moving Average Convergence Divergence.
    #
    # MACD is another technical indicator that can help
    # identify momentum and possible trend changes.
    #
    # We will compare the MACD line with its signal line.

    # Create a MACD indicator object.
    macd_indicator = MACD(

        # Use Bitcoin's closing prices.
        close=data["Close"],

        # Use a 12-period fast moving average.
        window_fast=12,

        # Use a 26-period slow moving average.
        window_slow=26,

        # Use a 9-period signal line.
        window_sign=9
    )

    # Calculate the MACD line.
    #
    # Store the result in a new column called "MACD".
    data["MACD"] = macd_indicator.macd()

    # Calculate the MACD signal line.
    #
    # Store the result in a new column called "MACD_SIGNAL".
    data["MACD_SIGNAL"] = macd_indicator.macd_signal()

    # Return the updated DataFrame.
    #
    # It now contains the original market data
    # plus our RSI and MACD calculations.
    return data


# ============================================================
# DECIDE WHAT TO DO
# ============================================================

# Define the trading_signal() function.
#
# Its job is to look at the indicators and decide:
#
# BUY
# SELL
# HOLD
#
# This is the strategy section of our bot.
def trading_signal(data):

    # Get the most recent candle.
    #
    # iloc[-1] means:
    # "Give me the last row in the DataFrame."
    current = data.iloc[-1]

    # Get the previous candle.
    #
    # iloc[-2] means:
    # "Give me the second-to-last row."
    #
    # We need this because we want to compare the previous
    # MACD values with the current MACD values.
    previous = data.iloc[-2]


    # Current RSI.

    # Get the RSI value from the most recent candle.
    rsi = current["RSI"]

    # Current MACD.

    # Get the MACD value from the most recent candle.
    macd = current["MACD"]

    # Current MACD signal.

    # Get the MACD signal-line value from the most recent candle.
    macd_signal = current["MACD_SIGNAL"]


    # Previous MACD values.

    # Get the MACD value from the previous candle.
    previous_macd = previous["MACD"]

    # Get the MACD signal-line value from the previous candle.
    previous_signal = previous["MACD_SIGNAL"]


    # --------------------------------------------------------
    # Check for a bullish MACD crossover
    # --------------------------------------------------------

    # A bullish crossover means:
    #
    # The MACD was at or below the signal line previously
    #
    # AND
    #
    # The MACD is now above the signal line.
    #
    # We interpret this as positive momentum
    # for this simple teaching strategy.

    # Create a Boolean variable.
    #
    # A Boolean can contain:
    #
    # True
    # or
    # False
    #
    # First we check whether the previous MACD
    # was less than or equal to the previous signal.
    bullish_macd = (
        previous_macd <= previous_signal

        # Then we check whether the current MACD
        # is now greater than the current signal.
        and macd > macd_signal
    )


    # --------------------------------------------------------
    # Check for a bearish MACD crossover
    # --------------------------------------------------------

    # A bearish crossover means:
    #
    # The MACD was at or above the signal line previously
    #
    # AND
    #
    # The MACD is now below the signal line.
    #
    # We interpret this as negative momentum
    # for this simple teaching strategy.

    bearish_macd = (
        # Check whether the previous MACD was
        # greater than or equal to the previous signal.
        previous_macd >= previous_signal

        # Check whether the current MACD is now
        # below the current signal.
        and macd < macd_signal
    )


    # --------------------------------------------------------
    # BUY RULE
    # --------------------------------------------------------

    # We buy when BOTH conditions are true:
    #
    # 1. RSI is below 30
    #
    # AND
    #
    # 2. MACD crosses upward.
    #
    # This is just an example strategy for teaching.
    #
    # It should not be interpreted as a guaranteed
    # profitable trading strategy.

    if rsi < 30 and bullish_macd:

        # If both conditions are True,
        # return the text "BUY".
        #
        # The main bot will receive this result
        # and call the buy_bitcoin() function.
        return "BUY"


    # --------------------------------------------------------
    # SELL RULE
    # --------------------------------------------------------

    # We sell when BOTH conditions are true:
    #
    # 1. RSI is above 70
    #
    # AND
    #
    # 2. MACD crosses downward.

    if rsi > 70 and bearish_macd:

        # If both conditions are True,
        # return the text "SELL".
        #
        # The main bot will receive this result
        # and call the sell_bitcoin() function.
        return "SELL"


    # Otherwise, neither the BUY nor SELL rules were triggered.
    #
    # Therefore, we return "HOLD".
    #
    # HOLD means the bot does not execute a trade.
    return "HOLD"


# ============================================================
# BUY BITCOIN
# ============================================================

# Define a function that simulates buying Bitcoin.
#
# The function receives the current Bitcoin price
# as the "price" argument.
def buy_bitcoin(price):

    # Python normally treats variables inside functions
    # as local variables.
    #
    # The "global" keyword tells Python that we want to
    # modify the cash variable created outside this function.
    global cash

    # Tell Python that we also want to modify
    # the global btc variable.
    global btc


    # Only use a percentage of our available cash.
    #
    # TRADE_SIZE = 0.25
    #
    # So if we have:
    #
    # $1,000
    #
    # The calculation becomes:
    #
    # $1,000 x 0.25 = $250
    #
    # This means the bot invests $250.
    dollars_to_invest = cash * TRADE_SIZE


    # Make sure we actually have money available.

    # Check whether the calculated amount is zero or negative.
    if dollars_to_invest <= 0:

        # If there is no money available,
        # stop running this function.
        return


    # Calculate how much Bitcoin we can buy.

    # Example:
    #
    # If we invest $250
    # and Bitcoin costs $100,000:
    #
    # $250 / $100,000
    #
    # = 0.0025 BTC
    bitcoin_bought = dollars_to_invest / price


    # Remove the money from our cash balance.

    # If we had $1,000 and invested $250:
    #
    # cash becomes:
    #
    # $1,000 - $250 = $750
    cash -= dollars_to_invest


    # Add Bitcoin to our account.

    # Add the Bitcoin we just purchased
    # to our existing BTC balance.
    btc += bitcoin_bought


    # Save the trade.

    # Create a dictionary containing information
    # about this simulated BUY.
    #
    # A dictionary stores information as:
    #
    # key : value
    trade = {

        # Record the current date and time.
        "time": pd.Timestamp.now(),

        # Store the fact that this was a BUY.
        "action": "BUY",

        # Store the Bitcoin price when we bought.
        "price": price,

        # Store how much Bitcoin we purchased.
        "bitcoin": bitcoin_bought,

        # Store how many dollars we invested.
        "dollars": dollars_to_invest
    }

    # Add the trade dictionary to our trade history list.
    #
    # This allows us to keep a record of every simulated trade.
    trade_history.append(trade)


    # Show the trade immediately.

    # Print a blank line to make the terminal easier to read.
    print()

    # Print a BUY banner.
    print("🚀 ================= BUY ================= 🚀")

    # Print the amount of Bitcoin purchased.
    print(
        f"Bought {bitcoin_bought:.8f} BTC"
    )

    # Print the price at which the simulated purchase occurred.
    print(
        f"Price: ${price:,.2f}"
    )

    # Print the total amount of cash used.
    print(
        f"Amount invested: ${dollars_to_invest:,.2f}"
    )

    # Print a divider below the BUY information.
    print("===========================================")


# ============================================================
# SELL BITCOIN
# ============================================================

# Define a function that simulates selling Bitcoin.
#
# The current Bitcoin price is passed into the function
# through the "price" parameter.
def sell_bitcoin(price):

    # Tell Python that we want to modify the global cash variable.
    global cash

    # Tell Python that we want to modify the global btc variable.
    global btc


    # If we don't own Bitcoin, there is nothing to sell.

    # Check whether our BTC balance is zero or less.
    if btc <= 0:

        # Tell the user that a SELL signal happened,
        # but the paper account does not own any Bitcoin.
        print("SELL signal received, but we don't own BTC.")

        # Stop this function because there is nothing to sell.
        return


    # Sell all of our Bitcoin.

    # Copy our entire BTC balance into a variable.
    #
    # This strategy sells 100% of the Bitcoin
    # currently owned.
    bitcoin_sold = btc


    # Calculate how much cash we receive.

    # Example:
    #
    # 0.002 BTC x $100,000
    #
    # = $200
    dollars_received = bitcoin_sold * price


    # Add the money back to our cash balance.

    # If we previously had $750 cash
    # and received $200 from selling:
    #
    # $750 + $200 = $950
    cash += dollars_received


    # We now own zero Bitcoin.

    # Set the BTC balance back to zero because
    # we just sold all of our Bitcoin.
    btc = 0


    # Save the trade.

    # Create a dictionary containing information
    # about this simulated SELL.
    trade = {

        # Record the current date and time.
        "time": pd.Timestamp.now(),

        # Store the fact that this was a SELL.
        "action": "SELL",

        # Store the Bitcoin price at the time of the sale.
        "price": price,

        # Store how much Bitcoin was sold.
        "bitcoin": bitcoin_sold,

        # Store how many dollars were received.
        "dollars": dollars_received
    }

    # Add the SELL trade to our trade history list.
    trade_history.append(trade)


    # Show the trade immediately.

    # Print a blank line for readability.
    print()

    # Print a SELL banner.
    print("🔴 ================= SELL ================= 🔴")

    # Print how much Bitcoin was sold.
    print(
        f"Sold {bitcoin_sold:.8f} BTC"
    )

    # Print the Bitcoin price used for the simulated sale.
    print(
        f"Price: ${price:,.2f}"
    )

    # Print the amount of cash received.
    print(
        f"Cash received: ${dollars_received:,.2f}"
    )

    # Print a divider below the SELL information.
    print("============================================")


# ============================================================
# SHOW PORTFOLIO
# ============================================================

# Define a function that calculates and displays
# our current paper trading portfolio.
#
# We give the function the current Bitcoin price
# because we need the price to calculate the value
# of the Bitcoin we own.
def show_portfolio(price):

    # Calculate the current value of our Bitcoin.

    # BTC owned x current BTC price
    #
    # gives us the current dollar value of our Bitcoin.
    bitcoin_value = btc * price


    # Total portfolio value =
    #
    # cash
    #
    # +
    #
    # value of Bitcoin

    # Add our cash and Bitcoin value together.
    portfolio_value = cash + bitcoin_value


    # Profit/loss compared with our original $1,000.

    # If the portfolio is now worth $1,100:
    #
    # $1,100 - $1,000
    #
    # = $100 profit
    profit_loss = portfolio_value - STARTING_MONEY


    # Percentage return.

    # First divide profit/loss by the starting amount.
    #
    # Then multiply by 100 to convert it into a percentage.
    return_percent = (
        profit_loss / STARTING_MONEY
    ) * 100


    # Print a blank line before the portfolio summary.
    print()

    # Print the portfolio heading.
    print("--------------- PORTFOLIO ----------------")

    # Print the amount of cash currently available.
    print(
        f"Cash:       ${cash:,.2f}"
    )

    # Print the amount of Bitcoin currently owned.
    print(
        f"BTC owned:  {btc:.8f}"
    )

    # Print the current dollar value of the Bitcoin.
    print(
        f"BTC value:  ${bitcoin_value:,.2f}"
    )

    # Print the total portfolio value.
    print(
        f"Portfolio:  ${portfolio_value:,.2f}"
    )

    # Print the dollar profit or loss.
    print(
        f"P/L:        ${profit_loss:,.2f}"
    )

    # Print the percentage return.
    print(
        f"Return:     {return_percent:.2f}%"
    )

    # Print a divider at the bottom of the portfolio summary.
    print("-------------------------------------------")


# ============================================================
# MAIN TRADING BOT
# ============================================================

# Define the main function that controls the entire bot.
#
# This function connects all of the previous functions together.
def run_bot():

    # Print a blank line before the startup information.
    print()

    # Print the top border of the startup message.
    print("==============================================")

    # Print the name of the trading bot.
    print("       🤖 BITCOIN AI PAPER TRADING BOT")

    # Print the bottom border of the title.
    print("==============================================")


    # Print the starting balance.
    print(
        f"Starting balance: ${STARTING_MONEY:,.2f}"
    )

    # Print the market the bot is trading.
    print(
        "Trading: BTC-USD"
    )

    # Print the current trading mode.
    #
    # IMPORTANT:
    # This tells us that the program is only simulating trades.
    print(
        "Mode: PAPER TRADING"
    )

    # Print a message explaining that the bot
    # is designed to keep running continuously.
    print(
        "Bot is running 24/7..."
    )

    # Print the bottom border of the startup information.
    print("==============================================")


    # --------------------------------------------------------
    # INFINITE LOOP
    # --------------------------------------------------------

    # This is what makes the bot autonomous.
    #
    # while True means:
    #
    # "Keep doing this forever."
    #
    # The bot will continue running until:
    #
    # 1. We manually stop it
    # 2. A serious unhandled problem stops the program
    #
    # In this code, most errors are caught below so
    # the bot can attempt to continue running.
    while True:

        # Start a try block.
        #
        # Everything inside this block is monitored for
        # unexpected errors.
        try:

            # Get Bitcoin market data.

            # Call our function that downloads Bitcoin data.
            #
            # The returned DataFrame is stored in "data".
            data = get_bitcoin_data()


            # Check whether the DataFrame contains no data.

            # data.empty returns True if the DataFrame
            # contains no rows.
            if data.empty:

                # Tell the user that market data
                # could not be retrieved.
                print("Could not get Bitcoin data.")

                # Wait 60 seconds before trying again.
                time.sleep(60)

                # Skip the rest of this loop iteration
                # and start the next iteration.
                continue


            # Calculate RSI and MACD.

            # Pass the market data into our indicator function.
            #
            # The function adds RSI and MACD columns
            # and returns the updated DataFrame.
            data = calculate_indicators(data)


            # Get the latest Bitcoin price.

            # Select the latest Close value
            # and convert it to a standard Python float.
            price = float(
                data.iloc[-1]["Close"]
            )


            # Get RSI.

            # Select the RSI value from the latest row
            # and convert it into a Python float.
            rsi = float(
                data.iloc[-1]["RSI"]
            )


            # Get MACD.

            # Select the MACD value from the latest row
            # and convert it into a Python float.
            macd = float(
                data.iloc[-1]["MACD"]
            )


            # Ask our strategy what to do.

            # Send the data into trading_signal().
            #
            # The function will return:
            #
            # "BUY"
            # "SELL"
            # or
            # "HOLD"
            signal = trading_signal(data)


            # ------------------------------------------------
            # DISPLAY MARKET INFORMATION
            # ------------------------------------------------

            # Print a blank line to make the terminal easier to read.
            print()

            # Print a divider between market checks.
            print("==============================================")


            # Print the current time.
            print(
                f"Time:       {pd.Timestamp.now()}"
            )

            # Print the latest Bitcoin price.
            print(
                f"BTC Price:  ${price:,.2f}"
            )

            # Print the current RSI.
            print(
                f"RSI:        {rsi:.2f}"
            )

            # Print the current MACD.
            print(
                f"MACD:       {macd:.4f}"
            )

            # Print the signal generated by our strategy.
            #
            # This will display:
            #
            # BUY
            # SELL
            # or
            # HOLD
            print(
                f"AI Signal:  {signal}"
            )


            # ------------------------------------------------
            # AUTOMATICALLY EXECUTE THE SIGNAL
            # ------------------------------------------------

            # Check whether the strategy returned "BUY".
            if signal == "BUY":

                # If the signal is BUY,
                # call the simulated buy function.
                #
                # We pass the current Bitcoin price
                # into the function.
                buy_bitcoin(price)


            # Check whether the signal is SELL.
            elif signal == "SELL":

                # If the signal is SELL,
                # call the simulated sell function.
                #
                # We pass the current Bitcoin price
                # into the function.
                sell_bitcoin(price)


            # If the signal was neither BUY nor SELL,
            # then it must be HOLD.
            else:

                # Tell the user that no trade occurred.
                print("⏸️ HOLD - No trade executed.")


            # ------------------------------------------------
            # SHOW UPDATED PORTFOLIO
            # ------------------------------------------------

            # Calculate and display the portfolio after
            # the current market check.
            show_portfolio(price)


            # ------------------------------------------------
            # SHOW TRADE COUNT
            # ------------------------------------------------

            # Print the total number of trades recorded.
            #
            # len(trade_history) tells us how many items
            # are currently inside the trade_history list.
            print(
                f"Total trades: {len(trade_history)}"
            )

            # Print a divider to finish this market check.
            print("==============================================")


            # ------------------------------------------------
            # WAIT BEFORE CHECKING AGAIN
            # ------------------------------------------------

            # Tell the user when the next market check will happen.
            print(
                f"Next check in "
                f"{CHECK_EVERY_SECONDS // 60} minutes..."
            )

            # Pause the program for the number of seconds
            # stored in CHECK_EVERY_SECONDS.
            #
            # Currently:
            #
            # CHECK_EVERY_SECONDS = 60
            #
            # Therefore:
            #
            # time.sleep(60)
            #
            # means the bot waits 60 seconds.
            time.sleep(CHECK_EVERY_SECONDS)


        # Catch a KeyboardInterrupt.
        #
        # This normally happens when the user presses:
        #
        # CTRL + C
        #
        # while the program is running.
        except KeyboardInterrupt:

            # Print a blank line before the shutdown message.
            print()

            # Tell the user that the bot was manually stopped.
            print("Bot stopped by user.")

            # Break exits the while True loop.
            #
            # This causes the bot to stop running.
            break


        # Catch any other unexpected error.
        #
        # The error itself is stored in the variable "error".
        except Exception as error:

            # Print the error so the user can see
            # what went wrong.
            print(
                f"Error: {error}"
            )

            # Tell the user that the bot will try again.
            print(
                "Trying again in 60 seconds..."
            )

            # Wait 60 seconds before attempting
            # another market check.
            time.sleep(60)


# ============================================================
# START THE BOT
# ============================================================

# Python automatically creates a special variable called
# __name__.
#
# When this file is run directly:
#
# __name__ == "__main__"
#
# This condition prevents run_bot() from automatically
# running if another Python file imports this file.
if __name__ == "__main__":

    # Start the trading bot by calling our main function.
    run_bot()


       🤖 BITCOIN AI PAPER TRADING BOT
Starting balance: $1,000.00
Trading: BTC-USD
Mode: PAPER TRADING
Bot is running 24/7...

Time:       2026-09-29 11:14:32.110906
BTC Price:  $82,949.11
RSI:        28.74
MACD:       -261.1405
AI Signal:  HOLD
⏸️ HOLD - No trade executed.

--------------- PORTFOLIO ----------------
Cash:       $1,000.00
BTC owned:  0.00000000
BTC value:  $0.00
Portfolio:  $1,000.00
P/L:        $0.00
Return:     0.00%
-------------------------------------------
Total trades: 0
Next check in 1 minutes...

Time:       2026-09-29 11:15:32.509401
BTC Price:  $82,995.93
RSI:        30.54
MACD:       -249.7151
AI Signal:  HOLD
⏸️ HOLD - No trade executed.

--------------- PORTFOLIO ----------------
Cash:       $1,000.00
BTC owned:  0.00000000
BTC value:  $0.00
Portfolio:  $1,000.00
P/L:        $0.00
Return:     0.00%
-------------------------------------------
Total trades: 0
Next check in 1 minutes...

Time:       2026-09-29 11:16:32.797830
BTC Price:  $83,002.46
RSI:    